
## 🎯 과제 2: 페르소나별 temperature 다르게 설정 (실무 빈도 ★★★★☆)

1교시에서 배웠죠. `temperature`는 페르소나의 성격과도 연결됩니다.

- **분석가**: `temperature=0` (정확하고 일관된 답변)
- **디자이너**: `temperature=0.8` (창의적이고 다양한 답변)

**요구사항**
- `PERSONAS` 사전 구조를 `{name: {"system": ..., "temperature": ...}}` 형태로 확장
- `ask()` 함수에서 페르소나별 `temperature`로 호출
- **힌트**: `ChatOpenAI(model=..., temperature=...)`을 페르소나마다 새로 만들거나, `.bind(temperature=...)`을 사용


In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")


✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [2]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": {"system": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.", "temperature": 0},
    "마케터": {"system": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.", "temperature": 0.5},
    "개발자": {"system": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.", "temperature": 0.2},
    "기획자": {"system": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.", "temperature": 0.4},
    "디자이너": {"system": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.", "temperature": 0.8},
}

def ask(persona_name: str, question: str) -> str:
    config = PERSONAS[persona_name]
    persona_chain = prompt | llm.bind(
        temperature=config["temperature"]
    )
    result = persona_chain.invoke({
        "role": config["system"],
        "question": question,
    })
    return result.content

# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

In [4]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

📖 명령어: /quit, /exit, /change <이름>, /persona, /help
[분석가] 1. **DataDive**: 데이터의 깊이를 탐험하며 분석 기술을 배울 수 있는 앱입니다.

2. **InsightQuest**: 통찰력을 얻기 위한 여정을 통해 데이터 분석의 기초를 익히는 플랫폼입니다.

3. **StatStart**: 통계의 기초부터 시작하여 데이터 분석의 세계로 나아가는 첫걸음 앱입니다.

4. **AnalyzeIt**: 데이터를 분석하는 방법을 직관적으로 배우고 실습할 수 있는 앱입니다.

5. **Learn2Data**: 데이터 분석을 배우는 과정을 쉽게 이해하고 적용할 수 있도록 돕는 학습 도구입니다.

✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
[디자이너] 장바구니에 상품을 담고도 구매하지 않는 사용자는 우리에게 중요한 단서입니다. 이 문제를 해결하기 위해 먼저 다음 세 가지를 우선적으로 고려해보세요.

1. **장바구니 리마인더 이메일 발송**
   - **우선순위:** ★★★★★
   - **이유:** 사용자가 장바구니에 담은 제품을 잊어버릴 수 있기 때문에, 장바구니 리마인더 이메일은 그들을 다시 끌어들이는 강력한 방법입니다. 이메일 안에 담긴 제품 이미지와 가격, 그리고 소정의 할인 쿠폰 등을 포함하면, 사용자가 다시 방문하도록 유도할 수 있습니다.

2. **장바구니 페이지 개선**
   - **우선순위:** ★★★★☆
   - **이유:** 장바구니 페이지가 복잡하거나 정보가 부족할 경우 사용자는 구매를 포기할 수 있습니다. 직관적인 UI, 명확한 가격 정보, 배송비 및 예상 배송일 등을 한눈에 보여주면 사용자가 마음의 결정을 쉽게 내릴 수 있습니다. 시각적으로 매력적인 장바구니 페이지는 구매 전환율을 높일 수 있습니다.

3. **신뢰성 강화**
   - **우선순위:** ★★★☆☆
   - **이유:** 사용자